# Пульсовые изменения удельного сопротивления

Ноутбук решает условную линейную обратную задачу для подписанных ансамблей
`33.03` в рабочей точке `33.01`. Для каждого дыхательного состояния оцениваются
временные функции $\Delta\rho_1(t)$ и $\Delta\rho_2(t)$.

Расчёт не запускается по амплитуде `max-min`, по некалиброванному каналу или по
старым встроенным параметрам. Результат относится к двуслойной модели и не
является прямым измерением локального кровенаполнения ткани.


## Линеаризация и допущения

В окрестности статической рабочей точки используется

$$
\Delta Z_L(t)=
\frac{\partial Z_L}{\partial\rho_1}\Delta\rho_1(t)+
\frac{\partial Z_L}{\partial\rho_2}\Delta\rho_2(t)+\varepsilon_L(t),
$$

где $L$ — размер сборки; $\Delta Z_L(t)$ — подписанный ансамблевый сигнал;
$\varepsilon_L(t)$ — совокупное несоответствие измерения и линеаризованной
модели. Размеры записаны последовательно, поэтому строки системы считаются
сопоставимыми только в рамках явно принятого допущения. Стандартные ошибки по
ударам не задают межзаписную ковариацию; решение выполняется без фиктивного
GLS-взвешивания.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import numpy as np

from two_layer_model import evaluate, geometry_from_size

REAL_MODE = os.environ.get("KALMYKOV_RUN_REAL", "0") == "1"


def sensitivity_matrix(sizes_m, rho1, rho2, h_m):
    rows = []
    for size_m in np.asarray(sizes_m, dtype=float):
        a, b = geometry_from_size(float(size_m))
        item = evaluate(float(rho1), float(rho2), float(h_m), a, b)
        rows.append([item.d_rho1, item.d_rho2])
    return np.asarray(rows)


def invert_waveforms(jacobian, waveforms_ohm):
    jacobian = np.asarray(jacobian, dtype=float)
    waveforms_ohm = np.asarray(waveforms_ohm, dtype=float)
    if jacobian.ndim != 2 or jacobian.shape[1] != 2 or waveforms_ohm.shape[0] != jacobian.shape[0]:
        raise ValueError("Несогласованные размеры Якобиана и ансамблей")
    rank = int(np.linalg.matrix_rank(jacobian))
    singular = np.linalg.svd(jacobian, compute_uv=False)
    if rank != 2:
        raise RuntimeError("Пульсовая линейная задача локально не имеет полного ранга")
    solution, _, _, _ = np.linalg.lstsq(jacobian, waveforms_ohm, rcond=None)
    residual = jacobian @ solution - waveforms_ohm
    return {
        "delta_rho1_ohm_m": solution[0],
        "delta_rho2_ohm_m": solution[1],
        "residual_ohm": residual,
        "rank": rank,
        "singular_values": singular,
        "condition": float(singular[0] / singular[-1]),
    }


In [ ]:
sizes_test = np.asarray([0.05, 0.06, 0.07, 0.08, 0.09, 0.11, 0.12, 0.13, 0.14])
j_test = sensitivity_matrix(sizes_test, 5.0, 18.0, 0.020)
t_test = np.arange(-0.10, 0.501, 0.005)
truth_test = np.vstack([
    0.001 * np.exp(-((t_test - 0.22) / 0.08) ** 2),
    -0.080 * np.exp(-((t_test - 0.30) / 0.09) ** 2),
])
wave_test = j_test @ truth_test
inverse_test = invert_waveforms(j_test, wave_test)
assert inverse_test["rank"] == 2
assert np.max(np.abs(inverse_test["delta_rho1_ohm_m"] - truth_test[0])) < 1e-10
assert np.max(np.abs(inverse_test["delta_rho2_ohm_m"] - truth_test[1])) < 1e-10
print("33.04 synthetic_self_test: passed")


In [ ]:
if not REAL_MODE:
    print("33.04 real_data_status: blocked_until_33.01_and_33.03_external_artifacts_exist")
else:
    config_value = os.environ.get("KALMYKOV_EXP02_CONFIG")
    if not config_value:
        raise RuntimeError("Задайте KALMYKOV_EXP02_CONFIG")
    config = json.loads(Path(config_value).expanduser().resolve().read_text(encoding="utf-8"))
    derived_root = Path(config["derived_root"]).expanduser().resolve()
    analysis_dir = derived_root / "exp02" / "analysis"
    static = json.loads((analysis_dir / "33.01_static.json").read_text(encoding="utf-8"))
    ensembles = json.loads((analysis_dir / "33.03_ensembles.json").read_text(encoding="utf-8"))
    if static.get("status") != "conditional_two_layer_estimate":
        raise RuntimeError("Неподходящий статус 33.01")
    if ensembles.get("status") != "accepted_input_conditional_ensembles":
        raise RuntimeError("Неподходящий статус 33.03")
    operator = ensembles.get("signal_operator", {})
    if operator.get("operator_status") != "delta_impedance_ohm_accepted" or operator.get("calibration_status") != "accepted":
        raise RuntimeError("Пульсовой оператор не принят")
    if static.get("config_sha256") != ensembles.get("config_sha256"):
        raise RuntimeError("33.01 и 33.03 построены по разным версиям конфигурации")

    grouped = {}
    for item in ensembles["ensembles"]:
        grouped.setdefault((item["subject_id"], item["mode"]), []).append(item)
    outputs = []
    for (subject_id, mode), items in sorted(grouped.items()):
        items.sort(key=lambda item: int(item["size_mm"]))
        static_subject = static["subjects"][subject_id]
        estimate = static_subject["estimate"]
        rho2 = estimate["rho2_inhale_ohm_m"] if mode == "задержка_вдох" else estimate["rho2_exhale_ohm_m"]
        sizes_mm = np.asarray([item["size_mm"] for item in items], dtype=float)
        if sizes_mm.tolist() != sorted(static_subject["sizes_mm"]):
            raise RuntimeError(f"Размеры 33.01 и 33.03 различаются: {subject_id}")
        time_grid = np.asarray(items[0]["time_from_r_s"], dtype=float)
        if any(not np.array_equal(time_grid, np.asarray(item["time_from_r_s"], dtype=float)) for item in items[1:]):
            raise RuntimeError("Ансамбли имеют разные временные сетки")
        waveforms = np.asarray([item["mean_ohm"] for item in items], dtype=float)
        jacobian = sensitivity_matrix(
            sizes_mm / 1000.0,
            estimate["rho1_ohm_m"], rho2, static_subject["h_m"],
        )
        result = invert_waveforms(jacobian, waveforms)
        outputs.append({
            "subject_id": subject_id,
            "mode": mode,
            "sizes_mm": sizes_mm.tolist(),
            "time_from_r_s": time_grid.tolist(),
            "delta_rho1_ohm_m": result["delta_rho1_ohm_m"].tolist(),
            "delta_rho2_ohm_m": result["delta_rho2_ohm_m"].tolist(),
            "residual_ohm": result["residual_ohm"].tolist(),
            "rank": result["rank"],
            "singular_values": result["singular_values"].tolist(),
            "condition": result["condition"],
        })
    artifact = {
        "schema_version": 1,
        "analysis": "33.04_pulse_delta_rho",
        "status": "conditional_linearized_two_layer_estimate",
        "assumptions": [
            "sequential_sizes_are_comparable",
            "linearization_about_33.01_working_point",
            "same_pulse_operator_for_all_sizes",
            "flat_two_layer_half_space_is_applicable",
        ],
        "limitations": [
            "no_inter_record_error_covariance",
            "within_record_SE_not_used_as_inter_record_weight",
            "ECG_synchrony_does_not_localize_source",
        ],
        "results": outputs,
    }
    out_path = analysis_dir / "33.04_delta_rho.json"
    out_path.write_text(json.dumps(artifact, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print("33.04 real_data_status: conditional_result_written", out_path)


## Критерий интерпретации

Полный ранг двух столбцов Якобиана является необходимым, но недостаточным
условием разделения вкладов. Большое число обусловленности означает сильное
усиление ошибок. Совпадение времени двух оценённых компонент может быть
следствием протекания модели и не доказывает общий физиологический источник.
Абсолютные значения нельзя передавать в ТТРКГ до независимой проверки
оператора пульсового канала и межзаписной неопределённости.
